# Cross-Domain QLoRA Fine-Tuning: Qwen2.5-0.8B

Fine-tune Qwen2.5-0.8B-Instruct on 3 text classification domains using QLoRA + Unsloth.

**Domains**: AG News (4 classes), GoEmotions (28 classes), LedGAR (100 classes)

**Setup**: Kaggle T4 GPU (16GB VRAM), ~2-3 hrs total

**Paper**: P8 (Entropy Predicts LLM Necessity) + P5 (Entropy-Aware Cascade)

In [ ]:
# Install dependencies
!pip install -q unsloth transformers datasets peft bitsandbytes accelerate trl scikit-learn

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

## 1. Load Datasets

In [ ]:
from datasets import load_dataset
import random
random.seed(42)

# =============================================
# CONFIGURATION — change DOMAIN here
# =============================================
DOMAIN = "ag_news"  # Options: "ag_news", "go_emotions", "ledgar"
SEED = 42
TRAIN_SIZE = 5000
TEST_SIZE = 1000
MAX_SEQ_LEN = 1024
NUM_EPOCHS = 3
LORA_RANK = 64
LORA_ALPHA = 128
LR = 2e-4
BATCH_SIZE = 4
GRAD_ACCUM = 8  # effective batch = 32
# =============================================

print(f"Domain: {DOMAIN} | Seed: {SEED} | Train: {TRAIN_SIZE}")

In [ ]:
from collections import Counter
import math

def load_domain_data(domain, train_size, test_size, seed):
    """Load and prepare domain-specific data."""
    
    if domain == "ag_news":
        ds = load_dataset("ag_news")
        label_names = ["World", "Sports", "Business", "Sci/Tech"]
        text_col, label_col = "text", "label"
        system_prompt = "You are a news topic classifier. Given a news article, classify it into exactly one category. Respond with ONLY the category name."
        
    elif domain == "go_emotions":
        ds = load_dataset("google-research-datasets/go_emotions", "simplified")
        label_names = [
            "admiration", "amusement", "anger", "annoyance", "approval",
            "caring", "confusion", "curiosity", "desire", "disappointment",
            "disapproval", "disgust", "embarrassment", "excitement", "fear",
            "gratitude", "grief", "joy", "love", "nervousness",
            "neutral", "optimism", "pride", "realization", "relief",
            "remorse", "sadness", "surprise"
        ]
        text_col, label_col = "text", "labels"
        system_prompt = "You are an emotion classifier. Given a text, identify the primary emotion. Respond with ONLY the emotion name."
        
    elif domain == "ledgar":
        ds = load_dataset("lex_glue", "ledgar")
        # LedGAR has 100 provision types
        label_names = ds["train"].features["label"].names
        text_col, label_col = "text", "label"
        system_prompt = "You are a legal document classifier. Given a contract provision, classify it into the correct provision type. Respond with ONLY the provision type name."
    
    else:
        raise ValueError(f"Unknown domain: {domain}")
    
    # Process labels
    def get_label_name(example):
        if domain == "go_emotions":
            # Multi-label: take first label
            labels = example[label_col]
            idx = labels[0] if labels else 20  # neutral
            return {"label_name": label_names[idx], "input_text": example[text_col]}
        else:
            return {"label_name": label_names[example[label_col]], "input_text": example[text_col]}
    
    train_ds = ds["train"].map(get_label_name)
    test_ds = ds["test" if "test" in ds else "validation"].map(get_label_name)
    
    # Subsample
    train_ds = train_ds.shuffle(seed=seed).select(range(min(train_size, len(train_ds))))
    test_ds = test_ds.shuffle(seed=seed).select(range(min(test_size, len(test_ds))))
    
    # Compute entropy
    label_counts = Counter(train_ds["label_name"])
    total = sum(label_counts.values())
    entropy = -sum((c/total) * math.log2(c/total) for c in label_counts.values())
    
    print(f"Domain: {domain}")
    print(f"Classes: {len(label_names)} | H(Y): {entropy:.2f} bits")
    print(f"Train: {len(train_ds)} | Test: {len(test_ds)}")
    print(f"Label dist (top 5): {label_counts.most_common(5)}")
    
    return train_ds, test_ds, label_names, system_prompt, entropy

train_ds, test_ds, label_names, system_prompt, entropy = load_domain_data(
    DOMAIN, TRAIN_SIZE, TEST_SIZE, SEED
)

## 2. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",  # Use 0.5B for safety on T4
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,  # auto
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

print(f"Model loaded. Trainable params: {model.print_trainable_parameters()}")

## 3. Format Data for Chat

In [ ]:
def format_chat(example):
    """Format as chat for Qwen."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Classify this text:\n\n{example['input_text']}"},
        {"role": "assistant", "content": example['label_name']}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_formatted = train_ds.map(format_chat)
print(f"Sample:\n{train_formatted[0]['text'][:500]}")

## 4. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

output_dir = f"/kaggle/working/{DOMAIN}_seed{SEED}"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_steps=10,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_strategy="epoch",
        seed=SEED,
        report_to="none",
    ),
)

import time
t0 = time.time()
trainer.train()
train_time = time.time() - t0
print(f"\nTraining complete in {train_time/60:.1f} minutes")

## 5. Evaluate

In [ ]:
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm
import json

FastLanguageModel.for_inference(model)

predictions = []
labels = []
raw_outputs = []

for i, example in enumerate(tqdm(test_ds, desc="Evaluating")):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Classify this text:\n\n{example['input_text']}"}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.01,
            do_sample=False,
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    
    predictions.append(response)
    labels.append(example['label_name'])
    raw_outputs.append({
        "input": example['input_text'][:200],
        "label": example['label_name'],
        "predict": response
    })

# Compute F1
strict_f1 = f1_score(labels, predictions, average='macro', zero_division=0)
accuracy = sum(1 for p, l in zip(predictions, labels) if p == l) / len(labels)

print(f"\n{'='*50}")
print(f"Domain: {DOMAIN} | H(Y): {entropy:.2f} bits")
print(f"Strict Macro F1: {strict_f1:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Train time: {train_time/60:.1f} min")
print(f"{'='*50}")

# Save results
results = {
    "domain": DOMAIN,
    "seed": SEED,
    "entropy": entropy,
    "strict_macro_f1": strict_f1,
    "accuracy": accuracy,
    "train_samples": len(train_ds),
    "test_samples": len(test_ds),
    "num_classes": len(label_names),
    "train_time_min": train_time / 60,
    "model": "Qwen2.5-0.5B-Instruct",
    "lora_rank": LORA_RANK,
}

with open(f"/kaggle/working/{DOMAIN}_results.json", "w") as f:
    json.dump(results, f, indent=2)

with open(f"/kaggle/working/{DOMAIN}_predictions.jsonl", "w") as f:
    for item in raw_outputs:
        f.write(json.dumps(item) + "\n")

print(f"\nSaved to /kaggle/working/{DOMAIN}_results.json")

In [ ]:
# Show classification report
print(classification_report(labels, predictions, zero_division=0))

In [ ]:
# Show some predictions
print("\n=== Sample Predictions ===")
for item in raw_outputs[:10]:
    status = "✅" if item['label'] == item['predict'] else "❌"
    print(f"{status} Label: {item['label']:20s} | Pred: {item['predict'][:30]}")

## 6. Save Adapter

Download the adapter weights for later use.

In [ ]:
# Save LoRA adapter
model.save_pretrained(f"/kaggle/working/{DOMAIN}_adapter")
tokenizer.save_pretrained(f"/kaggle/working/{DOMAIN}_adapter")
print(f"Adapter saved to /kaggle/working/{DOMAIN}_adapter")

# Zip for download
import shutil
shutil.make_archive(f"/kaggle/working/{DOMAIN}_adapter", 'zip', f"/kaggle/working/{DOMAIN}_adapter")
print(f"Download: /kaggle/working/{DOMAIN}_adapter.zip")